In [1]:
import os
os.environ["OMP_NUM_THREADS"] = "24"
import numpy as np
import numba as na
import matplotlib.pyplot as plt
from helpers import loss_channel_kraus,apply_kraus,coherent_state, gaussian_displacement, cat_ideal, compute_beta_norm, fidelity, catability

In [2]:
def kerr_norm_catable_operator (dim, parity, alpha, gamma, beta, norm):
    '''
    Computes the operator (2) in Fock basis.

    Parameters
    ----------
    dim : integer
        Dimension of the Hilbert space restriction to construct the operator on.

    Parameters (operator)
    ---------------------
    parity : np.integer (values -1, 1)
        Parity of the target Schrodinger state.
    alpha : np.complex
        Amplitude of the target Schrodinger state.
    gamma : np.floating
        Sensitivity of the non-Gaussian fringe detection.

    Returns
    -------
    np.ndarray
        The operator matrix.
    '''

    I = np.eye(dim)
    P = np.diag((- 1.0) ** np.arange(dim, dtype = np.float64))
    A = np.diag(np.sqrt(np.arange(1, dim, dtype = np.float64)), 1)
    D= gaussian_displacement(dim,beta)
    B = A @ A
    DPD = D @ P @ np.conjugate(D).T
    
    u = B.T - I * (np.conj(alpha) ** 2) 
    v = (B   - I * (alpha ** 2))
    w = gamma * (np.abs(norm)*I -parity * DPD)  #for kerr cats
    # w = gamma*(I - parity*P)                  #for parity cats 
    return u @ v + w

In [3]:
def compare_I(rho,dim, parity,aspace,gspace,eta,theta,threshold):
    ops = loss_channel_kraus(dim, eta)
    psi = apply_kraus(rho, ops)
    result = np.inf
    
    for aindex,gindex in np.ndindex(aspace.size, gspace.size):
        alpha_real = aspace[aindex].imag
        beta, norm = compute_beta_norm(alpha_real,parity,theta)
        
        if not (gspace[gindex] > 0):
            continue
        if not np.isfinite(threshold[aindex,gindex]):
            continue
        if not (threshold[aindex,gindex] > 0):
            continue
        
        op = kerr_norm_catable_operator (dim, parity, aspace[aindex], gspace[gindex], beta, norm)
        ksi = np.trace(psi @ op)
        ksi = np.real(ksi)
        value = ksi /threshold[aindex,gindex]
        
        if 0 <= value < result:
            result = value

    return np.real(result)


def compare_fid(rho,dim, parity, aspace, eta, theta,threshold):
    ops = loss_channel_kraus(dim, eta)
    psi = apply_kraus(rho, ops)
    result = np.inf
    
    for aindex,alpha in enumerate(aspace):
        if not np.isfinite(threshold[aindex]):
            continue
        if not (threshold[aindex] > 0):
            continue
        
        cat = cat_ideal(dim, alpha, theta)
        ksi = 1 - np.trace(cat @ psi)
        ksi = np.real(ksi)
        
        value = ksi/threshold[aindex]
        if 0 <= value < result:
            result = value

    return np.real(result)

In [4]:
def make_dataset (dim, rho, aspace, parity, bestcat,bestinf, gspace,tspace,theta):
    result = np.zeros(shape = (np.size(tspace),2), dtype = np.float64)
    for index, loss in enumerate(tspace):
        result[index,0] = compare_I(rho, dim, parity, aspace, gspace, loss, theta, bestcat)
        result[index,1] = compare_fid(rho, dim, parity, aspace, loss, theta, bestinf)
    return result

In [7]:
%%time
dim = 60
parity = 1
theta = np.pi/2

tspace = np.array([0.5,0.6,0.7,0.8,0.9,1.0 ])
np.save('results/comparison/tspace.npy', tspace)

aspace = np.load('results/comparison/aspace.npy')
gspace = np.load('results/comparison/gspace.npy')

bestcat = np.load('results/comparison/bestcat.npy')
bestinf = np.load('results/comparison/bestinf.npy')


tasklist = [
    (f'cat_i_0.5', cat_ideal(dim, 0.5j,theta)),
    (f'cat_i_1.0', cat_ideal(dim, 1.0j,theta)),
    (f'cat_i_1.5', cat_ideal(dim, 1.5j,theta)),
    (f'cat_i_2.0', cat_ideal(dim, 2.0j,theta)),
]
for label, rho in tasklist:
    result = make_dataset(dim,rho, aspace,parity,bestcat,bestinf,gspace,tspace,theta)
    np.save(f'results/comparison/{label}.npy', result)

CPU times: user 24min 20s, sys: 66.3 ms, total: 24min 20s
Wall time: 4min 56s
